In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive.mount("/content/drive", force_remount=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mounted at /content/drive


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── 1. Load the cleaned dataset ──────────────────────────────
print("Loading cleaned congestion dataset...")

congestion = pd.read_csv(
    "/content/drive/My Drive/data 400/data/congestion/congestion_cleaned_2022_2025.csv",
    low_memory=False
)

congestion['timestamp'] = pd.to_datetime(congestion['timestamp'], errors='coerce')

print(f"Loaded shape: {congestion.shape}")
print(f"Date range: {congestion['timestamp'].min()} → {congestion['timestamp'].max()}")
display(congestion.head(3))

Loading cleaned congestion dataset...
Loaded shape: (4621224, 23)
Date range: 2022-01-01 00:01:21 → 2025-12-31 23:50:25


,timestamp,region_id,speed_mph,region_name,bus_count,NUM_READS,HOUR,DAY_OF_WEEK,MONTH,DESCRIPTION,...,south_lat,north_lat,NW_LOCATION,SE_LOCATION,year,month,day_of_week,hour,year_month,congestion_level
0,2022-02-04 07:52:57,1,22.50,Rogers Park - West Ridge,36,744,7,6,2,North of Devon. Kedzie to Lake Shore,...,41.997946,42.026444,POINT (-87.709645 42.026444),POINT (-87.654561 41.997946),2022,2,Friday,7,2022-02,free_flow
1,2022-02-04 07:52:57,2,29.32,Far North West,39,"1,018",7,6,2,North of Montrose. East River to Cicero,...,41.960669,42.019100,POINT (-87.84621 42.0190998),POINT (-87.747456 41.960669),2022,2,Friday,7,2022-02,free_flow
2,2022-02-04 07:52:57,3,21.14,North Park-Albany-Linconl Sq,78,"1,578",7,6,2,Montrose to Devon. Cicero to Ravenswood,...,41.960669,41.997946,POINT (-87.747456 41.997946),POINT (-87.67459 41.960669),2022,2,Friday,7,2022-02,free_flow


In [ ]:
# ── 2. Remove impossible speed readings (sensor errors) ───────
before = len(congestion)
congestion = congestion[congestion['speed_mph'] <= 60]
removed = before - len(congestion)
print(f"\nRemoved {removed:,} rows with speed > 60 mph (sensor errors).")
print(f"Remaining: {len(congestion):,} rows")
print(f"Speed range after fix: {congestion['speed_mph'].min():.2f} – {congestion['speed_mph'].max():.2f} mph")



Removed 2,393 rows with speed > 60 mph (sensor errors).
Remaining: 4,618,831 rows
Speed range after fix: 0.20 – 60.00 mph


In [ ]:
# ── 3. Build date and 4-hour block columns ────────────────────
# Keep a real datetime date for sorting, plus the display string
congestion['date_dt']  = congestion['timestamp'].dt.date          # actual date for sorting
congestion['date']     = congestion['timestamp'].dt.strftime('%m/%d/%Y')  # display format

block_labels = {
    0: '00:00-03:59',
    1: '04:00-07:59',
    2: '08:00-11:59',
    3: '12:00-15:59',
    4: '16:00-19:59',
    5: '20:00-23:59',
}
congestion['time_block'] = (congestion['timestamp'].dt.hour // 4).map(block_labels)

print("\nTime block distribution:")
print(congestion['time_block'].value_counts().sort_index())




Time block distribution:
time_block
00:00-03:59    529619
04:00-07:59    779772
08:00-11:59    831295
12:00-15:59    833003
16:00-19:59    828516
20:00-23:59    816626
Name: count, dtype: int64


In [ ]:
# ── 4. Convert NUM_READS to numeric ──────────────────────────
congestion['NUM_READS'] = pd.to_numeric(
    congestion['NUM_READS'].astype(str).str.replace(',', ''), errors='coerce'
)


In [ ]:
# ── 5. Determine congestion_level per block ───────────────────
def most_common(series):
    return series.mode().iloc[0] if len(series) > 0 else None


In [ ]:
# ── 6. Aggregate to 4-hour blocks ────────────────────────────
print("\nAggregating to 4-hour blocks (this may take a moment)...")

agg_df = (
    congestion
    .groupby(['date_dt', 'date', 'time_block', 'region_id', 'region_name', 'DESCRIPTION'])
    .agg(
        avg_speed_mph    = ('speed_mph',        'mean'),
        avg_bus_count    = ('bus_count',         'mean'),
        avg_num_reads    = ('NUM_READS',          'mean'),
        congestion_level = ('congestion_level',   most_common),
        num_readings     = ('speed_mph',          'count'),
    )
    .reset_index()
)

# Round averaged columns
agg_df['avg_speed_mph'] = agg_df['avg_speed_mph'].round(2)
agg_df['avg_bus_count'] = agg_df['avg_bus_count'].round(1)
agg_df['avg_num_reads'] = agg_df['avg_num_reads'].round(1)



Aggregating to 4-hour blocks (this may take a moment)...


In [ ]:
# ── 7. Sort by REAL date (not string) then region then block ──
agg_df = agg_df.sort_values(
    ['date_dt', 'region_id', 'time_block']
).reset_index(drop=True)

# Drop the helper sort column — keep only the display date string
agg_df.drop(columns=['date_dt'], inplace=True)

# Rename DESCRIPTION for consistency
agg_df.rename(columns={'DESCRIPTION': 'description'}, inplace=True)

# Reorder columns cleanly
agg_df = agg_df[[
    'date',
    'time_block',
    'region_id',
    'region_name',
    'avg_speed_mph',
    'avg_bus_count',
    'avg_num_reads',
    'congestion_level',
    'description',
    'num_readings',
]]

print(f"\nAggregated shape: {agg_df.shape}")
print(f"(Expected ~29 regions × 6 blocks × ~1,460 days = ~254,040 rows)")
display(agg_df.head(12))



Aggregated shape: (214410, 10)
(Expected ~29 regions × 6 blocks × ~1,460 days = ~254,040 rows)


,date,time_block,region_id,region_name,avg_speed_mph,avg_bus_count,avg_num_reads,congestion_level,description,num_readings
0,01/01/2022,00:00-03:59,1,Rogers Park - West Ridge,27.73,5.8,100.1,free_flow,North of Devon. Kedzie to Lake Shore,10
1,01/01/2022,04:00-07:59,1,Rogers Park - West Ridge,28.46,10.1,224.3,free_flow,North of Devon. Kedzie to Lake Shore,21
2,01/01/2022,08:00-11:59,1,Rogers Park - West Ridge,25.58,17.5,363.7,free_flow,North of Devon. Kedzie to Lake Shore,23
3,01/01/2022,12:00-15:59,1,Rogers Park - West Ridge,23.38,20.1,429.9,free_flow,North of Devon. Kedzie to Lake Shore,17
4,01/01/2022,16:00-19:59,1,Rogers Park - West Ridge,20.40,19.6,436.5,free_flow,North of Devon. Kedzie to Lake Shore,23
5,01/01/2022,20:00-23:59,1,Rogers Park - West Ridge,22.44,13.3,284.6,free_flow,North of Devon. Kedzie to Lake Shore,21
6,01/01/2022,00:00-03:59,2,Far North West,28.98,6.1,68.9,free_flow,North of Montrose. East River to Cicero,10
7,01/01/2022,04:00-07:59,2,Far North West,31.26,16.3,370.2,free_flow,North of Montrose. East River to Cicero,22
8,01/01/2022,08:00-11:59,2,Far North West,31.27,21.8,508.6,free_flow,North of Montrose. East River to Cicero,23
9,01/01/2022,12:00-15:59,2,Far North West,28.92,22.2,504.4,free_flow,North of Montrose. East River to Cicero,17


In [ ]:
# ── 8. Sanity checks ──────────────────────────────────────────
print("\n── Congestion level distribution ──")
print(agg_df['congestion_level'].value_counts())

print("\n── Speed range (should be 0–60 mph) ──")
print(f"Min: {agg_df['avg_speed_mph'].min()} mph  |  Max: {agg_df['avg_speed_mph'].max()} mph")

print("\n── Rows per time block ──")
print(agg_df['time_block'].value_counts().sort_index())

print("\n── Date sort check (first 3 and last 3 dates) ──")
print(agg_df['date'].iloc[:3].tolist())
print(agg_df['date'].iloc[-3:].tolist())



── Congestion level distribution ──
congestion_level
free_flow    201876
moderate      12515
heavy            19
Name: count, dtype: int64

── Speed range (should be 0–60 mph) ──
Min: 3.62 mph  |  Max: 55.06 mph

── Rows per time block ──
time_block
00:00-03:59    34502
04:00-07:59    35856
08:00-11:59    36010
12:00-15:59    36101
16:00-19:59    35993
20:00-23:59    35948
Name: count, dtype: int64

── Date sort check (first 3 and last 3 dates) ──
['01/01/2022', '01/01/2022', '01/01/2022']
['12/31/2025', '12/31/2025', '12/31/2025']


In [ ]:
# ── 9. Export ─────────────────────────────────────────────────
output_path = "/content/drive/My Drive/data 400/data/congestion/updated_congestion_cleaned.csv"
agg_df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")
print(f"Rows: {len(agg_df):,}  |  Columns: {agg_df.shape[1]}")


Saved to: /content/drive/My Drive/data 400/data/congestion/updated_congestion_cleaned.csv
Rows: 214,410  |  Columns: 10
